# Part 2: A Simple Agent Based Model In Python

##### Authors: Bill Thompson (biltho@mpi.nl) and Limor Raviv (limor.raviv@mpi.nl)
Please let us know if you have any comments, suggestions or questions regarding this notebook.

---------------

## Summary
In this second tutorial, we will build our first simple network of agents using the commands in the first notebook.
We will create a network of multiple simple agents who interact in random dyads, and simulate changes in the community over multiple interactions. 
In each interaction, one agent (the producer) produces one of two possible vowel variants according to their existing representations. Then, the second agent (the listener) can either stick to their existing variant, or adapt to the producer by changing their vowel accordingly. The listener's behavior is based on biased "personalities": agents can be flexible (adapt to their partner) or stubborn (not adapt).
We repeat this process multiple times and see what happens to the variants in the population at large. We will try to answer questions like: Has one of the variants spread to the entire community? Does this depend on the community's size and inital structure? How many stubborn people must be present to prevent (or faciliate?) convergence? etc.

-------------- 


### 1. Setting the network
First, let's create lists containing the possible parameters for our agents. We will create a seperate list for the possible vowel representations and possible personalties our agents can have:

In [ ]:
# Setting the parameters

vowels = ['a', 'i']

personalities = ['F', 'S'] # F= Flexible, S=Stubborn

And let's write a simple function to create agents. An agent is just represented as a list with three values (an id, a vowel and a personality).

In [ ]:
def make_agent(i, vowel, personality):
    return [i, vowel, personality]

# For example, we can create a flexible agent with the vowel 'i' using our function

agent_one = make_agent(1, vowels[1], personalities[0])
print(agent_one)

Now, we can write functions that make populations of N agents (again, in the form of a list):

In [ ]:
# Create a function that generates a population of N identical agents with given parameters

def make_population_identical(N):
    
    population = []
    
    for i in range(N):
        
        agent = make_agent(i, vowels[1], personalities[0])
        
        population.append(agent)

    return population


# Call the function to make a population of 5 identical agents

population_test = make_population_identical(5)
print(population_test)

In [ ]:
# Create a function that generates of population of N agents with randomly selected parameters from each list
# using "random.choice()"

import random

def make_population_random(N):
    
    population = []
    
    for i in range(N):
        
        v = random.choice(vowels)
        
        p = random.choice(personalities)
        
        agent = make_agent(i, v, p)
        
        population.append(agent)

    return population

You can run the box of code below multiple times to make sure you are really getting random populations:

In [ ]:
# Call the function to make a population of 5 random agents
population = make_population_random(5)
print(population)

In [ ]:
# You can achieve the same goal using "random.int()" and using the index of the lists of possible parameters

def make_population(N):
    
    population = []
    
    for i in range(N):
        
        v = random.randint(0,1)
        
        p = random.randint(0,1)
        
        agent = make_agent(i, vowels[v], personalities[p])
        
        population.append(agent)

    return population

# Call the funtion and make a population of 8 random agents
# You can play with the numbers to make bigger or smaller populations
pop = make_population(8)
print(pop)

In [ ]:
# Create a function that calculates the proportion of agents with the variant 'a' in the population

def count(population):
    t = 0. # must be a float!     
    for agent in population:
        if agent[1] == 'a':
            t += 1            # The syntax =+ Adds 1 to t (or: t = t + 1)
    return t / len(population)

For a given population, we can now check how many agents are using each possible vowel variant. This is important, because later we'll also want to see how the proportion of each variant changes over the course of multiple interactions.

For this, we'll write a function that calculates the proportion of a given vowel in a population:

In [ ]:
# Call the funtion on a population of 20 random agents
# You can run this box multiple times to see the proportion in different populations of different sizes

prop_a = count(make_population(20))

print('The proportion of [a] in the population is', prop_a)

### 2. Interaction time!
We have a population, and now we want the agents to interact with each other. 

So first, we need to make a function that randomly selects two agents from the population:

In [ ]:
from numpy.random import choice

def choose_pair(population):
    i = random.randint(0, len(population) - 1) # phyton counts from 0, so pop(8) is an error
    j = random.randint(0, len(population) - 1)
    
    while i == j:
        j = random.randint(0, len(population) - 1) # make sure the same agent is not selected twice
        
    return population[i], population[j]


# And we'll test it to see that really does what we want
# You can run this box of code multiple times to make sure you are really getting random pairs

pop = make_population(8)
listener, producer = choose_pair(pop)

print('The population is', population)
print('This is the chosen pair', listener, producer)
print('The listener is', listener)
print('The producer is', producer)

Now, let's write a function that makes this pair "interact"!

If the producer and listener have the same vowel representation, nothing changes. If the agents have different vowels, then the listener's action depends on their prior personality: if they are stubborn, they will not change their vowel; but if they are flexible, they will update their vowel according to the producer.

So if the listener is flexible and has a different variant than the producer, we want to update the listener's vowel based on the producer's vowel.

To do this, we'll need to use a function called "deepcopy" to make a copy of the producer rather than pointing to the producer itself, because otherwise Python will have these two agents linked togeher forever. This is of course unwanted, since we want to update the listener only once based on a single interaction. Therefore, we'll use function called "deepcopy", which basically does what we want except for not linking the actual agents together.

In [ ]:
from copy import deepcopy

def interact_test(listener,producer): 
    
    if listener[1] == producer[1]:
        return listener # if the listener and producer have the same vowel, no change
    else:
        if listener[2]=='S':
            return listener # if the listener is stubborn, no change
        else:
            listener[1]=deepcopy(producer[1])
            return listener

You can check the output of the loop by running the code line below multiple times:

In [ ]:
randomlistener, randomproducer = choose_pair(make_population(8))

print('The listener is', randomlistener)
print('The producer is', randomproducer)

updated_listener = interact_test(randomlistener, randomproducer)

print('After interacting, the listener is',updated_listener)

So now we have a tested function that updates agents after interaction. Since we don't actually need the function to return the listener as output, we can change it to have no output, only to update the agents if needed:

In [ ]:
# Create a function that only updates agents using "pass" (which means do nothing in Python)

def interact(listener,producer): 
    
    if listener[1] == producer[1]:
        pass   # do nothing
    else:
        if listener[2]=='S':
            pass
        else:
            listener[1]=deepcopy(producer[1])

Ok, we're almost ready to run some simulations! So far, we've created a a few basic functions: 
1. Make Agent - creating an agent that has a vowel and a personality
2. Make Population - creating a population of agents using the function in (1)
3. Count - counting the proportion of agents with the same vowel in the population created by (2)
4. Choose Pair - choosing two agents out of the population created in (2)
5. Interact - implementing the interaxction between the two agents chosen in (4)

### 3. Simulation time!
The next step is to create a function that loops over the entire population based on a given number of desired interactions and check how many agents are using each possible vowel variant over time. 

In [ ]:
# Create a function that simulates a community of size N interacting randomly for K times       

def simulate(n, k):
    
    population = make_population(n)
    
    # print("Initial Population:", population)
    
    proportion = [] # make an empty list to keep track of the porportions after every interaction
    
    for i in range(k):
        
        pair = choose_pair(population) # choose a pair from the population
        
        interact(pair[0],pair[1])  # make the chosen pair interact
        
        proportion.append(count(population)) # track the proportion of the vowels in the population during intrtaction
    
    return population, proportion


Let's check that our function works by plotting the change in proportions of vowels over time. Again, you can run this multiple times to see what happens in different communities, and also change the numbers as you wish. 

In [ ]:
# Simulate 500 interctions between 20 agents 
new_population, proportion = simulate(20, 500)
print("Final Population:", new_population)

# Make a plot of the changes in proportion of 'a' over interactions 

%matplotlib inline 
#put plot in the notebook
import matplotlib.pyplot as plt # importing a plotting library
plt.plot(proportion)

# and add some details to the plot
plt.title('Changes in the proportion of [a] over time')
plt.ylabel('Proportion [a] users')
plt.xlabel('Time [No. of interactions]')
plt.ylim(0,1)

Keep in mind that bigger communities and more interactions will have more reliable results, but also will take more time to compute.

In [ ]:
# Simulate 5000 interctions between 200 agents 
new_population, proportion = simulate(200, 5000)
#print("Final Population:", new_population)

# Make a plot of the changes in proportion of 'a' over interactions 
print('   ')
print('Changes in the proportion of [a] over time')
plt.plot(proportion)
plt.ylim(0,1)

Wonderful, this works! But as you can see, there is a lot of variance in the results of different simulated populations.

To get a real picture of what's going on, we'll need to run multiple simulations. 

So let's write a new function that makes a bunch of simulations at once:

In [ ]:
# Create a function that runs s simulations of a community of size N interacting randomly for K times    

def batch_simulate(n,k,s):
    batch_proportions=[]
    for i in range(s):
        new_population, proportion = simulate(n, k)
        batch_proportions.append(proportion)
    return batch_proportions
        

You can check if the function works by plotting the results of this batch simulation. Again, you can run this multiple times and manipulate the size of the community (n), the number of interactions (k) or the number of simulations (s).

In [ ]:
# Make 20 simulations of 5000 interctions between 200 agents 
results = batch_simulate(200,5000,20)

plt.ylim(0,1)

for i in results:
    plt.plot(i)


### 4. Let's try to make a prediction and test it!
You made it! You've just run your first simple sound change simulation!

Perhaps you noticed that the trend is towards not converging: even after 5000 interactions, the proportion of one variant is still around 0.5, meaning the both vowels are prevelant in the popuation and there hasn't been a change towards one vowel. 

This is because we start out with random populations in which some stubborn agents never change. We predict that if there are at least 2 stubborn agents with *different* vowels, the flexible agents in the community will keep adapting to these *different* stubborn agents and never reaching stability and convergence. 

So we check ourselves: under which conditions can the community converge? How many stubborn people can there be in a population and still have convergence? 

To test this, we will modify our functions to create biased populations, where we control the number of stubborn agents:

In [ ]:
# Modify the function to make populations of N agents with a given number of stubborn agents (st)


def make_population_biased(N,st):
    
    population = []
    
    for i in range(st):
        
        v = random.randint(0,1)
        
        agent = make_agent(i, vowels[v], personalities[1])
        
        population.append(agent)
    
    for i in range(st, N):
        
        v = random.randint(0,1)
        
        agent = make_agent(i, vowels[v], personalities[0])
        
        population.append(agent)

    return population


# Modify the function so that it calls our biased population 

def simulate_biased(n, k, st):  #st=no. of stubborn
    
    population = make_population_biased(n,st)
    
    # print("Initial Population:", population)
    
    proportion = []
    
    for i in range(k):
        
        pair = choose_pair(population)
        
        interact(pair[0],pair[1])
        
        proportion.append(count(population))
    
    return population, proportion

Let's see what happens in  communities with no stubborn agents. Run this multiple times to see if any population reaches convergence (getting close to values of 1 or 0).

In [ ]:
# Run a simulation in a community with no stubborn agents

new_population, proportion = simulate_biased(200, 5000, 0)
print('Changes in the proportion of [a] over time')
plt.ylim(0,1)
plt.plot(proportion)


What about when there's one stubborn agent? Or two?

In [ ]:
# Run a simulation in a community with 1 stubborn agents

new_population, proportion = simulate_biased(200, 5000, 1)
print('Changes in the proportion of [a] over time')
plt.ylim(0,1)
plt.plot(proportion)

In [ ]:
# Run a simulation in a community with 2 stubborn agents

new_population, proportion = simulate_biased(200, 5000, 2)
print('Changes in the proportion of [a] over time')
plt.ylim(0,1)
plt.plot(proportion)

Again, there is too much variation. So let's run these simulations multiple times, unsing a wide range of proportions of stubborn people.

We'll run S simulations for each possible proportion of stubborn people.
Here, we'll check what happens when there are 0, 1, 2, 25%, 50% and 100% stubborn agents in the population.

In [ ]:
# Modify the function so it runs S simulations of each biased population

def batch_simulate_biased(n,k,s): #n-pop size, k=no. of interactions, s=no. of simulations for each bias
    
    all_results=[]
    
    possible_sts = [0, 1, 2, int(n / 4.), int(n / 2.), n]
    
    for possible_st in possible_sts:
        
        print(possible_st)
    
        current_results = []  # print the progress of the simulations 
    
        for i in range(s):

            new_population, proportion = simulate_biased(n, k, possible_st)
            current_results.append(proportion)
            
        all_results.append(current_results)
    
    return all_results

Now we can check how the proportion of stubborn agents affects convergence.

In [ ]:
# Run 20 simulations of each stubborness proportions in a community of 200 agents 
results = batch_simulate_biased(200,5000,20)

In [ ]:
# Plot the results of the simulations 

colors = ['green', 'blue', 'red', 'purple', 'orange', 'pink']
for j, st in enumerate(results):
    for simulation in st:
        plt.plot(simulation, color = colors[j], alpha = .5)

plt.ylim(0,1)
plt.xlabel('No. of Interactions')
plt.ylabel('Proportion Individuals Using [a]')       

What we see is that there are no clear patterns, except for when the population has 0-2 stubborn agents (in red, blue and green): in these cases, the population usually converges on one vowel. In all other cases, it's basically a mess.
If you want to practice your plotting skills, try adding a legend to this plot.

## Social network structure

##### Author: Lois Dona (lois.dona@mpi.nl) 

Now that you see patterns emerging based on number of stubborn/flexible agents, we can extend this to also look at the effect of different types of social networks.
If a network is densely connected, convergence may be easier than when agents have a hard time reaching each other. We can use the "networkx" library to create different types of networks and then modify our functions to make agents interact only with their neighbours in the network. You can find the documentation of the package here: https://networkx.org/en/.

In [ ]:
import networkx as nx

In this framework, social networks are presented as graphs. Agents are nodes within that graph. The connections between agents represent who can interact with who. 

In [ ]:
G = nx.Graph()

# Add individual nodes
G.add_node(1)
G.add_nodes_from([2, 3, 4, 7, 9])

print(G)
nx.draw(G, with_labels=True)

In [ ]:
# Add individual edges
G.add_edge(1, 2)
G.add_edge(3, 1)
G.add_edge(2, 4)
G.add_edge(4, 1)
G.add_edge(9, 1)

# Add multiple edges at once
G.add_edges_from([(1, 7), (2, 9)])

print(G)
nx.draw(G, with_labels=True)

In [ ]:
G.remove_node(3)             # Removes node 3
G.remove_edge(1, 2)          # Removes edge between node 1 and 2

print(G)
nx.draw(G, with_labels=True)

In [ ]:
# All nodes
print(G.nodes())            
# All edges
print(G.edges())             

# Number of nodes and edges
print(G.number_of_nodes()) 
print(G.number_of_edges())  

# Degree of a node (number of connections)
print(G.degree(2))          

# Neighbors of a node
print(list(G.neighbors(2)))

Now we know how to make graphs. We can adapt the choose_pair function we used before. Instead of representing the population as a list, we can now represent it in a graph format. An agent can only communicate with its neighbors in the graph.

In [ ]:
def choose_pair(graph):
    i = random.choice(list(graph.nodes()))
    j = random.choice(list(graph.neighbors(i)))
        
    return i, j

# Test the function to see that it really does what we want
G = nx.Graph()
G.add_edges_from([(1, 2), (1, 3), (2, 4), (3, 4), (4, 5)])
print(G)
nx.draw(G, with_labels=True)
i, j = choose_pair(G)
print('Chosen pair:', i, j)

Now that we know how to make graph, and we can let agents interact according to the graph structure, we can explore how patterns in vowel change look like under different social network conditions. In the following example, we explore what happens if there is a cluster of stubborn agents, and a cluster of flexible agents. 

In [ ]:
population = make_population_biased(8, 4) # instead, you could also hard-code the population by hand
print(population)

# put the agents in a network of your choice
G = nx.Graph()
G.add_edges_from([(0, 1), (1, 2), (1, 3), (2, 3), (0, 2), (0, 3), (3, 4), (4, 5), (4, 6), (4, 7), (5, 6), (5, 7), (6, 7)])
print(G)
nx.draw(G, with_labels=True)

In [ ]:
def simulate_graph(population, G, k):
    proportion = []
    
    for i in range(k):
        
        pair = choose_pair(G) # choose a pair from the graph
        
        interact(population[pair[0]], population[pair[1]])  # make the chosen pair interact
        
        proportion.append(count(population)) # track the proportion of the vowels in the population during intrtaction
    
    return population, proportion

In [ ]:
population, proportion = simulate_graph(population, G, 500)

print("Final population:", population)
print('Changes in the proportion of [a] over time')
plt.ylim(0,1)
plt.plot(proportion)

What you see is that the flexible agents all adapt to the vowel of the stubborn agent that is connected to their group, but the stubborn agents don't change their vowel use. 

### Exploration
Try out different kinds of networks and numbers of stubborn agents and see what happens! For example, you could try to compare networks with different degrees (number of neighbors of an agent) and numbers of stubborn agents, and see how this affects the speed of convergence on vowels.

In [ ]:
# Your code here